# 08 Fastlog

This notebook audits `tl.fastlog`, the sparse predicate recorder. A predicate is a small function that receives a `RecordContext` and decides whether to keep that event. Use fastlog when you know what you want to retain and do not need a full `Trace`.

We compare a full trace with a fastlog recording that keeps only ReLU operation events.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(37)


class FastlogNet(nn.Module):
    """Tiny MLP with linear and ReLU operations for sparse recording."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.stem = nn.Linear(3, 4)
        self.head = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the model.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Two-column output.
        """

        hidden = torch.relu(self.stem(x))
        return self.head(hidden)


def keep_relu_ops(ctx: tl.fastlog.RecordContext) -> bool:
    """Keep only ReLU operation events.

    Parameters
    ----------
    ctx:
        Fastlog event context.

    Returns
    -------
    bool
        Whether the event should be recorded.
    """

    return ctx.kind == "op" and ctx.layer_type == "relu"


model = FastlogNet().eval()
x = torch.ones(1, 3)
full_trace = tl.trace(model, x)
print(f"full trace layers: {full_trace.layer_labels}")
print(f"full trace layer count: {len(full_trace.layer_list)}")

`tl.fastlog.record` runs the model and keeps only predicate-selected records. The model still executes normally, but the result is a compact `Recording`, not a full graph trace.

In [ ]:
recording = tl.fastlog.record(model, x, keep_op=keep_relu_ops)
print(f"recording type: {type(recording).__name__}")
print(f"records kept: {len(recording.records)}")
print(f"operation records kept: {recording.n_ops}")
for record in recording.records:
    payload_shape = tuple(record.ram_payload.shape) if record.ram_payload is not None else None
    print(
        f"{record.ctx.label:18s} kind={record.ctx.kind:3s} "
        f"layer_type={record.ctx.layer_type:5s} payload_shape={payload_shape}"
    )

Returning a `CaptureSpec` gives more control over retained tensor payloads. In this build, per-record `metadata` is empty for these events, so the example prints the always-retained context fields (`shape`, `dtype`, and `parent_labels`) instead of claiming metadata payloads.

In [ ]:
def keep_relu_payload_linear_context(
    ctx: tl.fastlog.RecordContext,
) -> bool | tl.fastlog.CaptureSpec:
    """Keep ReLU payloads and linear context.

    Parameters
    ----------
    ctx:
        Fastlog event context.

    Returns
    -------
    bool | tl.fastlog.CaptureSpec
        Capture decision for the event.
    """

    if ctx.kind != "op":
        return False
    if ctx.layer_type == "relu":
        return tl.fastlog.CaptureSpec(save_out=True, save_metadata=True)
    if ctx.layer_type == "linear":
        return tl.fastlog.CaptureSpec(save_out=False, save_metadata=True)
    return False


mixed = tl.fastlog.record(model, x, keep_op=keep_relu_payload_linear_context)
for record in mixed.records:
    has_payload = record.ram_payload is not None
    print(
        f"{record.ctx.layer_type:6s} label={record.ctx.label:18s} "
        f"payload={has_payload} shape={record.ctx.shape} dtype={record.ctx.dtype} parents={record.ctx.parent_labels}"
    )

`dry_run` records the event stream without retaining payloads. This build previews predicate behavior by applying `repredicate(...)` to the dry-run contexts.

In [ ]:
dry = tl.fastlog.dry_run(model, x, keep_op=lambda ctx: True)
preview = dry.repredicate(other_keep_op=keep_relu_ops)
print(f"dry-run events observed: {len(dry.events)}")
print(f"predicate decisions with keeps: {sum(bool(decision) for decision in preview.decisions)}")
print(preview.to_pandas().head(6).to_string(index=False))

`tl.fastlog.halt(reason)` stops a recording early from inside a predicate. Records kept before the halt are preserved.

In [ ]:
def keep_ops_until_relu(ctx: tl.fastlog.RecordContext) -> bool:
    """Keep ops until the ReLU event, then halt.

    Parameters
    ----------
    ctx:
        Fastlog event context.

    Returns
    -------
    bool
        Whether to keep the event before halting.
    """

    if ctx.kind == "op" and ctx.layer_type == "relu":
        tl.fastlog.halt("stopped at relu")
    return ctx.kind == "op"


halted = tl.fastlog.record(model, x, keep_op=keep_ops_until_relu)
print(f"halted: {halted.halted}")
print(f"halt reason: {halted.halt_reason}")
print(
    f"kept before halt: {[(record.ctx.step_index, record.ctx.layer_type) for record in halted.records]}"
)

Fastlog is a sparse capture path. For graph structure, indexing, visualization, and full metadata, use `tl.trace`; for selected payloads in tight loops, use `tl.fastlog.record` or `tl.fastlog.Recorder`.

In [ ]:
with tl.fastlog.Recorder(model, keep_op=keep_relu_ops) as recorder:
    for scale in [1.0, 2.0]:
        recorder.log(x * scale)
loop_recording = recorder.recording
print(f"loop passes: {len(loop_recording.start_times)}")
print(f"loop kept records: {len(loop_recording.records)}")
print(f"loop labels: {[record.ctx.label for record in loop_recording.records]}")

> NOTE: Some glossary drafts describe a pass-count convenience field. This build exposes pass timing lists, so the notebook uses `len(recording.start_times)` for the number of recorded passes.

## 🔧 Sandbox

Mini-experiment: change `sandbox_layer_type`, `sandbox_save_out`, and `sandbox_preview_only`. Expected observation: selecting `linear` keeps two op records, selecting `relu` keeps one, and payload retention follows `sandbox_save_out`.

In [ ]:
sandbox_layer_type = "relu"
sandbox_save_out = True
sandbox_preview_only = False


def sandbox_keep(ctx: tl.fastlog.RecordContext) -> bool | tl.fastlog.CaptureSpec:
    """Keep sandbox-selected operation events.

    Parameters
    ----------
    ctx:
        Fastlog event context.

    Returns
    -------
    bool | tl.fastlog.CaptureSpec
        Capture decision for the event.
    """

    if ctx.kind == "op" and ctx.layer_type == sandbox_layer_type:
        return tl.fastlog.CaptureSpec(save_out=sandbox_save_out, save_metadata=True)
    return False


baseline_records = len(recording.records)
if sandbox_preview_only:
    sandbox_dry = tl.fastlog.dry_run(model, x, keep_op=lambda ctx: True).repredicate(
        other_keep_op=sandbox_keep
    )
    print(f"preview keeps: {sum(bool(decision) for decision in sandbox_dry.decisions)}")
else:
    sandbox_recording = tl.fastlog.record(model, x, keep_op=sandbox_keep)
    print(f"selected layer_type: {sandbox_layer_type}")
    print(f"records: {baseline_records} -> {len(sandbox_recording.records)}")
    for record in sandbox_recording.records:
        print(
            f"{record.ctx.label}: payload={record.ram_payload is not None} shape={record.ctx.shape}"
        )